In [ ]:
import ROOT as r
from itertools import permutations, combinations

# opens file and tree
f = r.TFile("actual_data/RunIISummer20UL17NanoAODv9-2_TTtoLNu2Q-1Jets-smeft_MTT-0to700_TuneCP5_13TeV_madgraphMLM-pythia8-2_NANOAODSIM106X_mc2017.root")
tree = f.Get("Events")

nEvents = tree.GetEntries()


# classes for muon, electron, jets

'''class MyMuon(r.TLorentzVector):
    def __init__(self, px=0, py=0, pz=0, e=0, iso=0.0, charge=0):
        super().__init__(px, py, pz, e)
        self.isolation = iso
        self.charge = charge''' #-------------px py pz that the cms_heptutorial dataset used



class MyMuon(r.TLorentzVector):
    def __init__(self, pt=0, eta=0, phi=0, mass=0, iso=0.0, charge=0):
        super().__init__()
        self.SetPtEtaPhiM(pt, eta, phi, mass)
        self.isolation = iso
        self.charge = charge
    
    
    def IsIsolated(self, relcut=0.1):
        if self.Pt() == 0:
            return False
        return self.isolation < relcut


'''class MyElectron(r.TLorentzVector):
    def __init__(self, px=0, py=0, pz=0, e=0, iso=0.0, charge=0):
        super().__init__(px, py, pz, e)
        self.isolation = iso
        self.charge = charge''' #-------------px py pz that the cms_heptutorial dataset used


class MyElectron(r.TLorentzVector):
    def __init__(self, pt=0, eta=0, phi=0, mass=0, iso=0.0, charge=0, cutBased=0):
        super().__init__()
        self.SetPtEtaPhiM(pt, eta, phi, mass)
        self.isolation = iso
        self.charge = charge
        self.cutBased = cutBased

    def IsIsolated(self, relcut=0.1):
        if self.Pt() == 0:
            return False
        return self.isolation < relcut

class MyJet(r.TLorentzVector):
    def __init__(self, pt=0, eta=0, phi=0, mass=0, btag=0.0, jetid=False):
        super().__init__()
        self.SetPtEtaPhiM(pt, eta, phi, mass)
        self.btag = btag
        self.jetid = jetid
        self.is_btagged = False

    def IsBTagged(self, threshold):
        return self.btag > threshold

    def HasJetID(self):
        return (self.jetid & 2) != 0
    
    
'''class MyJet(r.TLorentzVector):
    def __init__(self, px=0, py=0, pz=0, e=0, btag=0.0, jetid=False):
        super().__init__(px, py, pz, e)
        self.btag = btag
        self.jetid = jetid
        self.is_btagged = False''' #-------------px py pz that the cms_heptutorial dataset used


    
# histograms

# muons
h_NMuon   = r.TH1F("h_NMuon", "Number of isolated muons", 7, 0, 7)
h_Mmumu   = r.TH1F("h_Mmumu", "Invariant di-muon mass", 60, 60, 120)

# electrons
h_NElectron = r.TH1F("h_NElectron", "Number of isolated electrons", 7, 0, 7)
h_Mee       = r.TH1F("h_Mee", "Invariant di-electron mass", 60, 60, 120)

# jets
h_NJet   = r.TH1F("h_NJet", "Number of jets", 7, 0, 7)
h_NBJet  = r.TH1F("h_NBJet", "Number of b-jets", 7, 0, 7)

h_Jet1_Pt  = r.TH1F("h_Jet1_Pt", "p_{T} of leading jet", 50, 0, 250)
h_Jet2_Pt  = r.TH1F("h_Jet2_Pt", "p_{T} of subleading jet", 50, 0, 250)
h_Jet3_Pt  = r.TH1F("h_Jet3_Pt", "p_{T} of third jet", 50, 0, 250)
h_Jet4_Pt  = r.TH1F("h_Jet4_Pt", "p_{T} of 4th jet", 50, 0, 250)

h_Jet1_Eta = r.TH1F("h_Jet1_Eta", "#eta of leading jet", 50, -4, 4)
h_Jet2_Eta = r.TH1F("h_Jet2_Eta", "#eta of subleading jet", 50, -4, 4)
h_Jet3_Eta = r.TH1F("h_Jet3_Eta", "#eta of third jet", 50, -4, 4)
h_Jet4_Eta = r.TH1F("h_Jet4_Eta", "#eta of 4th jet", 50, -4, 4)

h_BJet1_Pt  = r.TH1F("h_BJet1_Pt", "p_{T} of leading b-jet", 50, 0, 250)
h_BJet2_Pt  = r.TH1F("h_BJet2_Pt", "p_{T} of subleading b-jet", 50, 0, 250)

h_BJet1_Eta = r.TH1F("h_BJet1_Eta", "#eta of leading b-jet", 50, -4, 4)
h_BJet2_Eta = r.TH1F("h_BJet2_Eta", "#eta of subleading b-jet", 50, -4, 4)


# 3 jets
h_Mtop_lep_3j_2b = r.TH1F("h_Mtop_lep_3j_2b", "Leptonic top (3j,2b)", 60, 0, 300)
h_Mtop_had_3j_2b = r.TH1F("h_Mtop_had_3j_2b", "Hadronic top (3j,2b)", 60, 0, 300)

h_Mtop_lep_3j_1b = r.TH1F("h_Mtop_lep_3j_1b", "Leptonic top (3j,1b)", 60, 0, 300)
h_Mtop_had_3j_1b = r.TH1F("h_Mtop_had_3j_1b", "Hadronic top (3j,1b)", 60, 0, 300)


# 4-6 jets
h_Mtop_lep_4pj_2b = r.TH1F("h_Mtop_lep_4pj_2b", "Leptonic top (≥4j,2b)", 60, 0, 300)
h_Mtop_had_4pj_2b = r.TH1F("h_Mtop_had_4pj_2b", "Hadronic top (≥4j,2b)", 60, 0, 300)

h_Mtop_lep_4pj_1b = r.TH1F("h_Mtop_lep_4pj_1b", "Leptonic top (≥4j,1b)", 60, 0, 300)
h_Mtop_had_4pj_1b = r.TH1F("h_Mtop_had_4pj_1b", "Hadronic top (≥4j,1b)", 60, 0, 300)

# met

h_MET     = r.TH1F("h_MET", "Missing E_{T}", 50, 0, 300)
h_METx    = r.TH1F("h_METx", "MET_x", 50, -300, 300)
h_METy    = r.TH1F("h_METy", "MET_y", 50, -300, 300)
h_METphi  = r.TH1F("h_METphi", "MET #phi", 50, -3.2, 3.2)

# hadronic and leptonic tops

h_Mtop_lep = r.TH1F("h_Mtop_lep", "Leptonic top mass", 60, 0, 300)
h_Mtop_had = r.TH1F("h_Mtop_had", "Hadronic top mass", 60, 0, 300)


# 1 btagged permutations

labels_3j_1b = [
    "blep_b_bhad_j1_W_j2",
    "blep_b_bhad_j2_W_j1",
    "blep_j1_bhad_b_W_j2",
    "blep_j1_bhad_j2_W_b",
    "blep_j2_bhad_b_W_j1",
    "blep_j2_bhad_j1_W_b"
]

h_Mtop_lep_3j_1b_cases = {}
h_Mtop_had_3j_1b_cases = {}

for label in labels_3j_1b:
    h_Mtop_lep_3j_1b_cases[label] = r.TH1F(
        f"h_Mtop_lep_3j_1b_{label}",
        f"Leptonic top: {label};M [GeV];Events",
        60, 0, 300
    )

    h_Mtop_had_3j_1b_cases[label] = r.TH1F(
        f"h_Mtop_had_3j_1b_{label}",
        f"Hadronic top: {label};M [GeV];Events",
        60, 0, 300
    )

h_Mtop_lep_4pj_cases = {}
h_Mtop_had_4pj_cases = {}

h_Mtop_lep_4pj_1b_cases = {}
h_Mtop_had_4pj_1b_cases = {}

h_Mtop_lep_4pj_2b_cases = {}
h_Mtop_had_4pj_2b_cases = {}

def get_case_hist(hist_dict, name, title):
    if name not in hist_dict:
        hist_dict[name] = r.TH1F(
            name,
            f"{title};M [GeV];Events",
            60, 0, 300
        )
        hist_dict[name].Sumw2()

    return hist_dict[name]

# weighted errors
for h in [
    h_NMuon, h_Mmumu,
    h_NElectron, h_Mee,
    h_NJet, h_NBJet,
    h_Jet1_Pt, h_Jet2_Pt, h_Jet3_Pt, h_Jet4_Pt,
    h_Jet1_Eta, h_Jet2_Eta, h_Jet3_Eta, h_Jet4_Eta,
    h_BJet1_Pt, h_BJet2_Pt,
    h_BJet1_Eta, h_BJet2_Eta,
    h_MET, h_METx, h_METy, h_METphi,
    h_Mtop_lep, h_Mtop_had,
    h_Mtop_lep_3j_2b, h_Mtop_had_3j_2b,
    h_Mtop_lep_3j_1b, h_Mtop_had_3j_1b,
    h_Mtop_lep_4pj_2b, h_Mtop_had_4pj_2b,
    h_Mtop_lep_4pj_1b, h_Mtop_had_4pj_1b
    ]:
    h.Sumw2()


# analysis cuts
'''
MuonRelIsoCut     = 0.1
MuonPtCut         = 25.0

ElectronRelIsoCut = 0.1
ElectronPtCut     = 25.0

JetPtCut          = 30.0
BTagThreshold     = 0.5 '''

cuts = {
    "Muon": {
        "pt_min": 30.0,
        "eta_max": 2.4,
        "iso_max": 0.15
    },
    "Electron": {
        "pt_min": 30.0,
        "eta_max": 2.4,
        "cutBased": 4,
        "iso_max": 0.15
        
    },
    "Jets": {
        "pt_min": 30.0,
        "eta_max": 2.4,
        "btag": 0.3040
    },
    "MET": {
        "pt_min": 20.0
    }
}

cuts["Trigger"] = {
    "muon": ["HLT_IsoMu27"],
    "electron": ["HLT_Ele27_WPTight_Gsf", "HLT_Ele32_WPTight_Gsf"]
}

cutflow = {
    "total": 0,

    # triggers
    "pass_mu_trigger": 0,
    "pass_ele_trigger": 0,

    # lepton selection
    "exactly_1_lepton": 0,

    # MET
    "pass_MET": 0,

    # jets
    "3jets": 0,
    "4plus_jets": 0,

    # b-tag categories
    "3j_2b": 0,
    "3j_1b": 0,
    "4pj_2b": 0,
    "4pj_1b": 0
}

# event loop
selectedevents = 0
for event in range(nEvents):

    cutflow["total"] += 1

    tree.GetEntry(event)
    weight = tree.Generator_weight
    

    #MET cuts
    if not all([
        tree.Flag_goodVertices,
        tree.Flag_globalSuperTightHalo2016Filter,
        tree.Flag_HBHENoiseFilter,
        tree.Flag_HBHENoiseIsoFilter,
        tree.Flag_EcalDeadCellTriggerPrimitiveFilter,
        tree.Flag_BadPFMuonFilter,
        tree.Flag_BadPFMuonDzFilter,
        tree.Flag_eeBadScFilter,
        tree.Flag_ecalBadCalibFilter
        ]):
        continue


    # trigger cuts
    passes_mu_trigger = any(
        getattr(tree, trig, False) for trig in cuts["Trigger"]["muon"]
        )

    passes_ele_trigger = any(
        getattr(tree, trig, False) for trig in cuts["Trigger"]["electron"]
    )   

    if passes_mu_trigger:
        cutflow["pass_mu_trigger"] += 1

    if passes_ele_trigger:
        cutflow["pass_ele_trigger"] += 1

    # muons

    muons = []
    for mu_idx in range(tree.nMuon):
        mu = MyMuon(
            tree.Muon_pt[mu_idx],
            tree.Muon_eta[mu_idx],
            tree.Muon_phi[mu_idx],
            tree.Muon_mass[mu_idx],   
            tree.Muon_miniPFRelIso_all[mu_idx],
            tree.Muon_charge[mu_idx] #unsure
            )
        '''mu = MyMuon(
            tree.Muon_Px[mu_idx],
            tree.Muon_Py[mu_idx],
            tree.Muon_Pz[mu_idx],
            tree.Muon_E[mu_idx],
            tree.Muon_Iso[mu_idx],
            tree.Muon_Charge[mu_idx]
        )'''
        muons.append(mu)

    #iso_muons = [m for m in muons if m.IsIsolated(MuonRelIsoCut)]

    iso_muons = [
    m for m in muons
    if m.Pt() > cuts["Muon"]["pt_min"]
    and abs(m.Eta()) < cuts["Muon"]["eta_max"]
    and m.isolation < cuts["Muon"]["iso_max"]
    ]



    iso_muons = sorted(iso_muons, key=lambda m: m.Pt(), reverse=True)
    h_NMuon.Fill(len(iso_muons), weight)

    #if len(iso_muons) >= 2 and iso_muons[0].Pt() > MuonPtCut:
        #dimu = iso_muons[0] + iso_muons[1]
        #h_Mmumu.Fill(dimu.M(), weight)


    # electrons
    electrons = []
    for ele_idx in range(tree.nElectron):
        ele = MyElectron(
            tree.Electron_pt[ele_idx],
            tree.Electron_eta[ele_idx],
            tree.Electron_phi[ele_idx],
            tree.Electron_mass[ele_idx],   
            tree.Electron_miniPFRelIso_all[ele_idx],  
            tree.Electron_charge[ele_idx],
            tree.Electron_cutBased[ele_idx]
            )
        '''ele = MyElectron(
            tree.Electron_Px[ele_idx],
            tree.Electron_Py[ele_idx],
            tree.Electron_Pz[ele_idx],
            tree.Electron_E[ele_idx],
            tree.Electron_Iso[ele_idx],
            tree.Electron_Charge[ele_idx]
        )'''
        electrons.append(ele)

    
    iso_electrons = [
        e for e in electrons
        if e.Pt() > cuts["Electron"]["pt_min"]
        and abs(e.Eta()) < cuts["Electron"]["eta_max"]
        and e.cutBased >= cuts["Electron"]["cutBased"]
        and e.isolation < cuts["Electron"]["iso_max"] 
        ]

    #iso_electrons = [e for e in electrons if e.IsIsolated(ElectronRelIsoCut)]
    iso_electrons = sorted(iso_electrons, key=lambda e: e.Pt(), reverse=True)

    h_NElectron.Fill(len(iso_electrons), weight)

    #if len(iso_electrons) >= 2 and iso_electrons[0].Pt() > ElectronPtCut:
    #   diele = iso_electrons[0] + iso_electrons[1]
    #  h_Mee.Fill(diele.M(), weight)


    # MET from tree
    MET  = tree.MET_pt
    phi  = tree.MET_phi


    METx = MET * r.TMath.Cos(phi)
    METy = MET * r.TMath.Sin(phi)

    
    '''METx = tree.MET_px
    METy = tree.MET_py

    MET  = (METx**2 + METy**2)**0.5
    METphi = r.TMath.ATan2(METy, METx)''' #CMS HEP Tutorial stuff

    h_MET.Fill(MET, weight)
    h_METx.Fill(METx, weight)
    h_METy.Fill(METy, weight)
    h_METphi.Fill(phi, weight)

    n_iso_mu  = len(iso_muons)
    n_iso_ele = len(iso_electrons)


    '''exactly_one_lepton = (
        (n_iso_mu == 1 and n_iso_ele == 0) or
        (n_iso_ele == 1 and n_iso_mu == 0)
    )''' #-----this is without the trigger
    if n_iso_mu == 1 and n_iso_ele == 0:
        if not passes_mu_trigger:
            continue
        selected_lepton = iso_muons[0]

    elif n_iso_ele == 1 and n_iso_mu == 0:
        if not passes_ele_trigger:
            continue
        selected_lepton = iso_electrons[0]

    else:
        continue
    cutflow["exactly_1_lepton"] += 1
    passes_MET = MET > cuts["MET"]["pt_min"]

    '''if not exactly_one_lepton:
        continue
    cutflow["1_lepton"] += 1'''

    if not passes_MET:
        continue
    cutflow["pass_MET"] += 1

    # jets

    jets = []
    for jet_idx in range(tree.nJet):
        jet = MyJet(
            tree.Jet_pt[jet_idx],
            tree.Jet_eta[jet_idx],
            tree.Jet_phi[jet_idx],
            tree.Jet_mass[jet_idx],    
            tree.Jet_btagDeepFlavB[jet_idx],    
            tree.Jet_jetId[jet_idx] 
            )
        
        '''jet = MyJet(
            tree.Jet_Px[jet_idx],
            tree.Jet_Py[jet_idx],
            tree.Jet_Pz[jet_idx],
            tree.Jet_E[jet_idx],
            tree.Jet_btag[jet_idx],
            tree.Jet_ID[jet_idx]
        )'''
        jets.append(jet)

    
    def deltaR(obj1, obj2):
        return obj1.DeltaR(obj2)

    good_jets = [
    j for j in jets
    if j.Pt() > cuts["Jets"]["pt_min"]
    and abs(j.Eta()) < cuts["Jets"]["eta_max"]
    and j.HasJetID()
    and deltaR(j, selected_lepton) > 0.4
]

    #good_jets = [j for j in jets if j.HasJetID() and j.Pt() > JetPtCut]
    good_jets = sorted(good_jets, key=lambda j: j.Pt(), reverse=True)
    
    

    # flagging b-tagged jets
    for j in good_jets:
        # j.is_btagged = j.IsBTagged(BTagThreshold)
        j.is_btagged = j.IsBTagged(cuts["Jets"]["btag"])

    #bjets = [j for j in good_jets if j.IsBTagged(BTagThreshold)] - old usage before fagging btagged jets
    bjets = [j for j in good_jets if j.is_btagged]
    bjets = sorted(bjets, key=lambda j: j.Pt(), reverse=True)

    n_jets = len(good_jets)
    n_bjets = len(bjets)

    
    h_NJet.Fill(len(good_jets), weight)
    h_NBJet.Fill(len(bjets), weight)

    # Leading jets
    if len(good_jets) > 0:
        h_Jet1_Pt.Fill(good_jets[0].Pt(), weight)
        h_Jet1_Eta.Fill(good_jets[0].Eta(), weight)

    if len(good_jets) > 1:
        h_Jet2_Pt.Fill(good_jets[1].Pt(), weight)
        h_Jet2_Eta.Fill(good_jets[1].Eta(), weight)

    if len(good_jets) > 2:
        h_Jet3_Pt.Fill(good_jets[2].Pt(), weight)
        h_Jet3_Eta.Fill(good_jets[2].Eta(), weight)
    if len(good_jets) > 3:
        h_Jet4_Pt.Fill(good_jets[3].Pt(), weight)
        h_Jet4_Eta.Fill(good_jets[3].Eta(), weight)

    # Leading b-jets
    if len(bjets) > 0:
        h_BJet1_Pt.Fill(bjets[0].Pt(), weight)
        h_BJet1_Eta.Fill(bjets[0].Eta(), weight)

    if len(bjets) > 1:
        h_BJet2_Pt.Fill(bjets[1].Pt(), weight)
        h_BJet2_Eta.Fill(bjets[1].Eta(), weight)

    #3 jets
    if n_jets == 3:
        cutflow["3jets"] += 1


        # 2 btagged
        if n_bjets >= 2:
            cutflow["3j_2b"] += 1
            b1, b2 = bjets[0], bjets[1]

            light_jets = sorted(
                [j for j in good_jets if not j.is_btagged],
                key=lambda j: j.Pt(),
                reverse=True
            )[:3]

            

            if len(light_jets) < 1:
                continue

            j1 = light_jets[0]

            b_permutations = [(b1, b2), (b2, b1)]

            for b_lep, b_had in b_permutations:
                top_lep = selected_lepton + b_lep
                h_Mtop_lep_3j_2b.Fill(top_lep.M(), weight)

                top_had = j1 + b_had   
                h_Mtop_had_3j_2b.Fill(top_had.M(), weight)
        
        # 1 btagged

        elif n_bjets == 1:
            cutflow["3j_1b"] += 1
            b1 = bjets[0]

            light_jets = sorted(
                [j for j in good_jets if not j.is_btagged],
                key=lambda j: j.Pt(),
                reverse=True
                )[:3]
            
            if len(light_jets) < 2:
                continue
            
            candidate_jets = [b1, light_jets[0], light_jets[1]]
            candidate_names = ["b", "j1", "j2"]

            for perm in permutations(range(3)):
                i_blep, i_bhad, i_W = perm

                label = (
                    f"blep_{candidate_names[i_blep]}_"
                    f"bhad_{candidate_names[i_bhad]}_"
                    f"W_{candidate_names[i_W]}"
                )

                b_lep_candidate = candidate_jets[i_blep]
                b_had_candidate = candidate_jets[i_bhad]
                W_jet_candidate = candidate_jets[i_W]

                top_lep = selected_lepton + b_lep_candidate
                top_had = b_had_candidate + W_jet_candidate

                h_Mtop_lep_3j_1b_cases[label].Fill(top_lep.M(), weight)
                h_Mtop_had_3j_1b_cases[label].Fill(top_had.M(), weight)
            
    #4-6 jets
    elif 4 <= n_jets <= 6:
        cutflow["4plus_jets"] += 1

        light_jets = sorted(
            [j for j in good_jets if not j.is_btagged],
            key=lambda j: j.Pt(),
            reverse=True
        )

        # at least 2 btagged
        if n_bjets >= 2:
            cutflow["4pj_2b"] += 1

            if len(light_jets) < 2:
                continue

            for i_blep, i_bhad in permutations(range(n_bjets), 2):

                for i_W1, i_W2 in combinations(range(len(light_jets)), 2):

                    label = (
                        f"{n_jets}j_"
                        f"blep_b{i_blep + 1}_"
                        f"bhad_b{i_bhad + 1}_"
                        f"W_j{i_W1 + 1}j{i_W2 + 1}"
                    )

                    b_lep = bjets[i_blep]
                    b_had = bjets[i_bhad]

                    W_had = (
                        light_jets[i_W1]
                        + light_jets[i_W2]
                    )

                    top_lep = selected_lepton + b_lep
                    top_had = W_had + b_had

                    h_lep = get_case_hist(
                        h_Mtop_lep_4pj_2b_cases,
                        f"h_Mtop_lep_4pj_2b_{label}",
                        f"Leptonic top: {label}"
                    )

                    h_had = get_case_hist(
                        h_Mtop_had_4pj_2b_cases,
                        f"h_Mtop_had_4pj_2b_{label}",
                        f"Hadronic top: {label}"
                    )

                    h_lep.Fill(top_lep.M(), weight)
                    h_had.Fill(top_had.M(), weight)
                    h_Mtop_lep_4pj_2b.Fill(top_lep.M(), weight)
                    h_Mtop_had_4pj_2b.Fill(top_had.M(), weight)

        # 1 btagged
        elif n_bjets == 1:
            cutflow["4pj_1b"] += 1

            if len(light_jets) < 3:
                continue

            tagged_b = bjets[0]

            # Tagged jet is the leptonic b
            for i_bhad in range(len(light_jets)):

                remaining = [
                    i for i in range(len(light_jets))
                    if i != i_bhad
                ]

                for i_W1, i_W2 in combinations(remaining, 2):

                    label = (
                        f"{n_jets}j_"
                        f"blep_btag_"
                        f"bhad_j{i_bhad + 1}_"
                        f"W_j{i_W1 + 1}j{i_W2 + 1}"
                    )

                    top_lep = selected_lepton + tagged_b
                    top_had = (
                        light_jets[i_bhad]
                        + light_jets[i_W1]
                        + light_jets[i_W2]
                    )

                    get_case_hist(
                        h_Mtop_lep_4pj_1b_cases,
                        f"h_Mtop_lep_4pj_1b_{label}",
                        f"Leptonic top: {label}"
                    ).Fill(top_lep.M(), weight)

                    get_case_hist(
                        h_Mtop_had_4pj_1b_cases,
                        f"h_Mtop_had_4pj_1b_{label}",
                        f"Hadronic top: {label}"
                    ).Fill(top_had.M(), weight)

            # Tagged jet is the hadronic b
            for i_blep in range(len(light_jets)):

                remaining = [
                    i for i in range(len(light_jets))
                    if i != i_blep
                ]

                for i_W1, i_W2 in combinations(remaining, 2):

                    label = (
                        f"{n_jets}j_"
                        f"blep_j{i_blep + 1}_"
                        f"bhad_btag_"
                        f"W_j{i_W1 + 1}j{i_W2 + 1}"
                    )

                    top_lep = selected_lepton + light_jets[i_blep]
                    top_had = (
                        tagged_b
                        + light_jets[i_W1]
                        + light_jets[i_W2]
                    )

                    get_case_hist(
                        h_Mtop_lep_4pj_1b_cases,
                        f"h_Mtop_lep_4pj_1b_{label}",
                        f"Leptonic top: {label}"
                    ).Fill(top_lep.M(), weight)

                    get_case_hist(
                        h_Mtop_had_4pj_1b_cases,
                        f"h_Mtop_had_4pj_1b_{label}",
                        f"Hadronic top: {label}"
                    ).Fill(top_had.M(), weight)



# histograms

print("\n=== CUTFLOW ===")
for key, val in cutflow.items():
    print(f"{key:15s}: {val}")

c_muon = r.TCanvas("c_muon", "Muon Histograms", 800, 600)
h_NMuon.Draw()
c_muon.Update()

c_jets = r.TCanvas("c_jets", "Jet Histograms", 800, 600)
h_NJet.Draw()
c_jets.Update()

c_jets_pt = r.TCanvas("c_jets_pt", "Jet pT comparison", 800, 600)

h_Jet1_Pt.SetLineColor(1)
h_Jet2_Pt.SetLineColor(2)
h_Jet3_Pt.SetLineColor(3)
h_Jet4_Pt.SetLineColor(4)

h_Jet1_Pt.Draw("HIST")
h_Jet2_Pt.Draw("HIST SAME")
h_Jet3_Pt.Draw("HIST SAME")
h_Jet4_Pt.Draw("HIST SAME")

leg = r.TLegend(0.7,0.7,0.9,0.9)
leg.AddEntry(h_Jet1_Pt, "Leading jet", "l")
leg.AddEntry(h_Jet2_Pt, "Subleading jet", "l")
leg.AddEntry(h_Jet3_Pt, "Third jet", "l")
leg.AddEntry(h_Jet4_Pt, "Fourth jet", "l")
leg.Draw()

c_jets_pt.Update()

c_nbjet = r.TCanvas("c_nbjet", "Number of b-jets", 800, 600)
h_NBJet.Draw("HIST")
c_nbjet.Update()

c_bjet_pt = r.TCanvas("c_bjet_pt", "B-jet pT comparison", 800, 600)

h_BJet1_Pt.SetLineColor(2)  # red
h_BJet2_Pt.SetLineColor(4)  # blue

h_BJet1_Pt.Draw("HIST")
h_BJet2_Pt.Draw("HIST SAME")

leg_bjet = r.TLegend(0.7, 0.7, 0.9, 0.9)
leg_bjet.AddEntry(h_BJet1_Pt, "Leading b-jet", "l")
leg_bjet.AddEntry(h_BJet2_Pt, "Subleading b-jet", "l")
leg_bjet.Draw()

c_bjet_pt.Update()

c_electron = r.TCanvas("c_electron", "Electron Histograms", 800, 600)
h_NElectron.Draw()
c_electron.Update()

c_met = r.TCanvas("c_met", "MET Histograms", 800, 600)
h_MET.Draw()
c_met.Update()

# 3 jets

c_top_3j_cases = r.TCanvas(
    "c_top_3j_cases",
    "3j 1b permutations",
    1200, 800
)
c_top_3j_cases.Divide(3, 2)

legends_3j = []

for i, label in enumerate(labels_3j_1b):
    c_top_3j_cases.cd(i + 1)

    h_had = h_Mtop_had_3j_1b_cases[label]
    h_lep = h_Mtop_lep_3j_1b_cases[label]

    h_had.SetLineColor(r.kRed)
    h_lep.SetLineColor(r.kBlue)

    h_had.Draw("HIST")
    h_lep.Draw("HIST SAME")

    legend = r.TLegend(0.55, 0.72, 0.88, 0.88)
    legend.AddEntry(h_had, "Hadronic candidate", "l")
    legend.AddEntry(h_lep, "Leptonic candidate", "l")
    legend.Draw()

    legends_3j.append(legend)

# 2 btagged
c_top_3j = r.TCanvas("c_top_3j", "Top Reconstruction (3 jets)", 800, 600)
c_top_3j.Divide(2,2)

c_top_3j.cd(1)
h_Mtop_lep_3j_2b.Draw()

c_top_3j.cd(2)
h_Mtop_had_3j_2b.Draw()

c_top_3j_cases.Update()
c_top_3j.Update()

#4 jets
c_top_4pj_cases = r.TCanvas(
    "c_top_4pj_cases",
    ">=4j 1b cases",
    1200, 800
)
c_top_4pj_cases.Divide(3, 2)

legends_4pj = []

for i, (had_name, h_had) in enumerate(
    h_Mtop_had_4pj_1b_cases.items()
):
    if i >= 6:
        break

    lep_name = had_name.replace(
        "h_Mtop_had_4pj_1b_",
        "h_Mtop_lep_4pj_1b_"
    )

    if lep_name not in h_Mtop_lep_4pj_1b_cases:
        continue

    h_lep = h_Mtop_lep_4pj_1b_cases[lep_name]

    c_top_4pj_cases.cd(i + 1)

    h_had.SetLineColor(r.kRed)
    h_lep.SetLineColor(r.kBlue)

    h_had.Draw("HIST")
    h_lep.Draw("HIST SAME")

    legend = r.TLegend(0.55, 0.72, 0.88, 0.88)
    legend.AddEntry(h_had, "Hadronic candidate", "l")
    legend.AddEntry(h_lep, "Leptonic candidate", "l")
    legend.Draw()

    legends_4pj.append(legend)

c_top_4pj_cases.Update()

import os
os.makedirs("plots", exist_ok=True)

c_muon.SaveAs("plots/h_muon.png")
c_jets.SaveAs("plots/h_jets.png")
c_jets_pt.SaveAs("plots/h_jets_pt.png")
c_nbjet.SaveAs("plots/h_nbjet.png")
c_bjet_pt.SaveAs("plots/h_bjet_pt.png")
c_electron.SaveAs("plots/h_electron.png")
c_met.SaveAs("plots/h_met.png")
c_top_3j_cases.SaveAs("plots/h_top_3j.png")
c_top_4pj_cases.SaveAs("plots/h_top_4pj.png")
#c_top.SaveAs("plots/h_top.png")


Warning in <TClass::Init>: no dictionary for class edm::Hash<1> is available
Warning in <TClass::Init>: no dictionary for class edm::ProcessHistory is available
Warning in <TClass::Init>: no dictionary for class edm::ProcessConfiguration is available
Warning in <TClass::Init>: no dictionary for class edm::ParameterSetBlob is available
Warning in <TClass::Init>: no dictionary for class pair<edm::Hash<1>,edm::ParameterSetBlob> is available



=== CUTFLOW ===
total          : 197164
pass_mu_trigger: 38934
pass_ele_trigger: 33116
exactly_1_lepton: 60383
pass_MET       : 55203
3jets          : 15776
4plus_jets     : 29264
3j_2b          : 5874
3j_1b          : 7849
4pj_2b         : 16438
4pj_1b         : 10864


Info in <TCanvas::Print>: png file plots/h_muon.png has been created
Info in <TCanvas::Print>: png file plots/h_jets.png has been created
Info in <TCanvas::Print>: png file plots/h_jets_pt.png has been created
Info in <TCanvas::Print>: png file plots/h_nbjet.png has been created
Info in <TCanvas::Print>: png file plots/h_bjet_pt.png has been created
Info in <TCanvas::Print>: png file plots/h_electron.png has been created
Info in <TCanvas::Print>: png file plots/h_met.png has been created
Info in <TCanvas::Print>: png file plots/h_top_3j.png has been created
Info in <TCanvas::Print>: png file plots/h_top_4pj.png has been created
